# Market environment: a guided tour

Env notebook (Francesco). It shows the trading environment in `Reference/envs/` doing what [`interface.md`](interface.md) says it does.

Everything here is a **mechanical check of the simulator**. Nothing is tuned and nothing is selected, so none of these numbers is a result of the project. The scripted policies on SPY are run on the whole file only to see that the arithmetic matches the prices.

In [1]:
import json
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root))

from Reference.envs import TradingEnv, run_policy

pd.options.display.float_format = "{:,.5f}".format


def score(returns, actions, cost_bps, k):
    history = run_policy(TradingEnv(returns, k=k, cost_bps=cost_bps, days="all"), actions)
    return sum(row["net_return"] for row in history), sum(row["trade"] for row in history)

## 1. The toy example, scored by the environment

Seven closing prices. With `k = 1` the first move is only history and five days are traded: +10%, +10%, −10%, 0, +10%. The table should match the one worked out by hand in `interface.md`.

In [2]:
prices = [100.0, 100.0, 110.0, 121.0, 108.9, 108.9, 119.79]
toy = np.diff(np.log(prices))

policies = {
    "always flat": [0, 0, 0, 0, 0],
    "always long": [1, 1, 1, 1, 1],
    "switch every day": [1, 0, 1, 0, 1],
    "oracle (long on the up days)": [1, 1, 0, 0, 1],
}
rows = []
for name, actions in policies.items():
    free, changes = score(toy, actions, 0.0, k=1)
    paid, _ = score(toy, actions, 10.0, k=1)
    rows.append({"policy": name, "total at zero cost": free, "changes": changes, "total at 10 bps": paid,
                 "difference": free - paid})
display(pd.DataFrame(rows).set_index("policy"))
print("always long at zero cost, as a wealth multiple:", round(math.exp(rows[1]["total at zero cost"]), 6),
      "| price ratio 119.79 / 100 =", 119.79 / 100)

,total at zero cost,changes,total at 10 bps,difference
policy,,,,
always flat,0.00000,0,0.00000,0.00000
always long,0.18057,1,0.17957,0.00100
switch every day,0.08526,5,0.08026,0.00500
oracle (long on the up days),0.28593,3,0.28293,0.00300


always long at zero cost, as a wealth multiple: 1.1979 | price ratio 119.79 / 100 = 1.1979


## 2. The same policies on SPY

Data's file from the feasibility check, adjusted closes, `k = 20`. Two things must hold whatever the prices are:

- always long at zero cost multiplies wealth by exactly the ratio of the last to the first adjusted close of the traded window;
- switching every day loses exactly `cost × number of changes` compared with the same policy at zero cost.

The oracle is told each day's return in advance. It is here to show what one bar of look-ahead is worth.

In [3]:
spy_file = root / "results" / "matteo_feasibility" / "20261115T212629605692Z" / "SPY_checked.csv"
spy = pd.read_csv(spy_file, parse_dates=["Date"])
closes = spy["Adj Close"].to_numpy()
spy_returns = np.diff(np.log(closes))        # position j holds the move into the (j+1)-th row of the file
K = 20
traded = spy_returns[K:]                      # what "all" trades: every day from k on

spy_policies = {
    "always flat": np.zeros(len(traded), dtype=int),
    "always long": np.ones(len(traded), dtype=int),
    "switch every day": (np.arange(len(traded)) % 2 == 0).astype(int),
    "oracle": (traded > 0).astype(int),
}
rows = []
for name, actions in spy_policies.items():
    row = {"policy": name}
    for cost_bps in (0.0, 2.0, 5.0):
        total, changes = score(spy_returns, actions.tolist(), cost_bps, K)
        row["changes"] = changes
        row[f"wealth x at {cost_bps:g} bps"] = math.exp(total)
    rows.append(row)
pd.options.display.float_format = "{:,.3f}".format
display(pd.DataFrame(rows).set_index("policy"))

print(f"{len(traded)} days traded, {spy['Date'].iloc[K + 1].date()} to {spy['Date'].iloc[-1].date()}")
print("price ratio over the traded window:", round(closes[-1] / closes[K], 6))
free, changes = score(spy_returns, spy_policies["switch every day"].tolist(), 0.0, K)
paid, _ = score(spy_returns, spy_policies["switch every day"].tolist(), 2.0, K)
print(f"switch every day: zero cost minus 2 bps = {free - paid:.6f} | 0.0002 x {changes} changes = {0.0002 * changes:.6f}")

,changes,wealth x at 0 bps,wealth x at 2 bps,wealth x at 5 bps
policy,,,,
always flat,0,1.000,1.000,1.000
always long,1,9.180,9.178,9.176
switch every day,4222,3.657,1.572,0.443
oracle,2122,"11,114,968.042","7,270,991.555","3,846,996.339"


4222 days traded, 2010-02-03 to 2026-11-13
price ratio over the traded window: 9.180283
switch every day: zero cost minus 2 bps = 0.844400 | 0.0002 x 4222 changes = 0.844400


## 3. What the agent sees

The observation is the last `k` returns in percent followed by the current position. The first decision below is taken at the close of the 21st row of the file and earns the return of the 22nd.

In [4]:
dates = spy["Date"].dt.date.astype(str).to_numpy()[1:]   # one label per return: the day it is realised
env = TradingEnv(spy_returns, k=K, cost_bps=2.0, days="all", dates=dates)
obs, _ = env.reset()
print("observation length:", len(obs), "= 20 returns + position")
print("last three returns seen (%):", np.round(obs[17:20], 3), "| position:", obs[20])
print("they are the returns of:", dates[K - 3 : K])
_, reward, _, _, info = env.step(1)
print(f"going long earns the return of {info['date']}: gross {info['gross_return']:+.5f}, cost {info['cost']:.5f}, net {reward:+.5f}")

peeking = TradingEnv(spy_returns, k=K, cost_bps=2.0, days="all", peek=True)
obs_peek, _ = peeking.reset()
print("with the peek on, one more number:", round(float(obs_peek[21]), 3), "% = the return about to be earned:",
      round(100 * spy_returns[K], 3), "%")

observation length: 21 = 20 returns + position
last three returns seen (%): [-1.093  1.543  1.203] | position: 0.0
they are the returns of: ['2010-01-29' '2010-02-01' '2010-02-02']
going long earns the return of 2010-02-03: gross -0.00500, cost 0.00020, net -0.00520
with the peek on, one more number: -0.5 % = the return about to be earned: -0.5 %


## 4. The leakage check, live

Replace every return from the day about to be traded onwards with made-up numbers. With the peek off the observation must not move at all; with the peek on it must.

In [5]:
rng = np.random.default_rng(0)
tampered = spy_returns.copy()
tampered[K:] = rng.normal(0, 0.05, len(tampered) - K)

for peek in (False, True):
    honest, _ = TradingEnv(spy_returns, k=K, cost_bps=2.0, days="all", peek=peek).reset()
    changed, _ = TradingEnv(tampered, k=K, cost_bps=2.0, days="all", peek=peek).reset()
    print(f"peek={peek!s:5} | observation identical after rewriting the future: {np.array_equal(honest, changed)}")

peek=False | observation identical after rewriting the future: True
peek=True  | observation identical after rewriting the future: False


## 5. Can an agent learn here? A synthetic market

`feasibility.py` trains PPO and DQN on **random numbers**, so the right answers are known. With the peek on, a working agent must get close to the oracle. With the peek off there is nothing to learn, and on a fresh series the agent must end near zero whatever it scored in training.

`train` and `test` are total log returns; the test series is a fresh draw of 1,000 days on which the oracle earns 3.54 and luck alone moves a total by about ±0.63.

In [6]:
rows = []
for summary_file in sorted((root / "results" / "francesco_feasibility").glob("*/summary.json")):
    summary = json.loads(summary_file.read_text())
    for run in summary["runs"]:
        rows.append({"steps": run["budget"], "algorithm": run["algorithm"], "peek": run["peek"],
                     "returns x": run["obs_scale"], "rewards x": run["reward_scale"],
                     "seconds": run["training_seconds"], "train": run["train"]["total_log_return"],
                     "test": run["test"]["total_log_return"], "share of oracle": run["test_share_of_oracle"]})
pd.options.display.float_format = "{:,.2f}".format
display(pd.DataFrame(rows).sort_values(["steps", "algorithm", "peek"], ascending=[False, True, False]).reset_index(drop=True))

,steps,algorithm,peek,returns x,rewards x,seconds,train,test,share of oracle
0,100000,DQN,True,100,100,23.78,7.11,3.28,0.93
1,100000,DQN,True,1,100,24.17,7.51,3.53,1.00
2,100000,DQN,True,100,1,23.46,5.97,2.55,0.72
3,100000,DQN,False,100,100,23.41,4.61,-0.36,-0.10
4,100000,PPO,True,100,100,34.74,7.39,3.48,0.98
5,100000,PPO,True,1,100,34.54,7.41,3.48,0.98
6,100000,PPO,True,100,1,34.49,7.38,3.47,0.98
7,100000,PPO,False,100,100,34.57,2.87,-0.45,-0.13
8,3000,DQN,True,100,100,0.80,7.36,3.45,0.97
9,3000,DQN,True,1,100,0.80,0.00,0.00,0.00


### What the table says

- **Peek on, 100,000 steps:** both agents reach 93 to 100% of the oracle on a series they have never seen (72% for DQN when rewards are not rescaled). The environment and the training code work together.
- **Peek off, 100,000 steps:** both agents look very profitable on the series they trained on and make nothing on the fresh one. On random numbers that is the only honest outcome, and it is the project's point in miniature.
- **3,000 steps:** with returns in percent and rewards multiplied by 100 the agents have already learned; with either scaling missing they have not. Scaling changes how fast they learn, not what there is to learn.

One seed per row: enough to say the pipeline works, not enough to rank anything.